In [1]:
# ── Cell 0: HF login ──────────────────────────────────────────────────────
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
login(token=UserSecretsClient().get_secret("HF_TOKEN1"))
print("HF login: OK")

# ── Cell 1: Install ─────────────────────────────────────────────────────────
!pip install peft bitsandbytes transformers accelerate -q

# ── Cell 2: Imports ──────────────────────────────────────────────────────────
import os, re, warnings
import torch
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import LlavaForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
warnings.filterwarnings("ignore")
print("Imports: OK")

# ── Cell 3: Config ───────────────────────────────────────────────────────────
VAL_TXT  = "/kaggle/input/datasets/softhell/med-vqa-2021/data/val/label.txt"
VAL_IMGS = "/kaggle/input/datasets/softhell/med-vqa-2021/data/val/images"
EVAL_CSV = "/kaggle/working/llava_med_metrics.csv"
MODEL_ID = "chaoyinshe/llava-med-v1.5-mistral-7b-hf"
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
MAX_NEW_TOKENS = 40

# <<< CHANGE THIS to wherever the checkpoint actually mounted >>>
adapter_path = "/kaggle/input/datasets/julfikarrahamanasif/vhcghvfg/checkpoint_epoch3"

SYSTEM_PROMPT = (
    "You are a radiology AI assistant. "
    "When asked about a medical image, respond with the diagnosis "
    "or abnormality name only — no explanations, no full sentences. "
    "Keep your answer to 1-6 words maximum."
)

LLAVA_CHAT_TEMPLATE = (
    "{% for message in messages %}"
        "{% if message['role'] == 'system' %}"
            "{% for item in message['content'] %}"
                "{% if item['type'] == 'text' %}{{ item['text'] }}\n{% endif %}"
            "{% endfor %}"
        "{% elif message['role'] == 'user' %}"
            "USER: "
            "{% for item in message['content'] %}"
                "{% if item['type'] == 'image' %}<image>\n{% endif %}"
                "{% if item['type'] == 'text' %}{{ item['text'] }}\n{% endif %}"
            "{% endfor %}"
        "{% elif message['role'] == 'assistant' %}"
            "ASSISTANT: "
            "{% for item in message['content'] %}"
                "{% if item['type'] == 'text' %}{{ item['text'] }}{% endif %}"
            "{% endfor %}"
            "{{ eos_token }}"
        "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}ASSISTANT: {% endif %}"
)

print(f"Device : {DEVICE}")
print(f"Adapter: {adapter_path}")

def normalise(text):
    # 1. Convert to lowercase
    text = str(text).lower().strip()
    # 2. Strip out all punctuation
    text = re.sub(r"[^\w\s]", " ", text)
    # 3. Handle whitespace (prepares for clean tokenization)
    return re.sub(r"\s+", " ", text).strip()

def get_synonyms(answer):
    return [normalise(s) for s in str(answer).split(",") if normalise(s)]

def exact_match_any(pred, answer):
    # Strict match: Is the prediction exactly one of the synonyms?
    return int(normalise(pred) in get_synonyms(answer))

def relaxed_match_any(pred, answer):
    # Relaxed match: Does the prediction contain the clinical keyword?
    pred_norm = normalise(pred)
    for syn in get_synonyms(answer):
        if syn in pred_norm:
            return 1
    return 0

def compute_bleu_ngrams(pred_tokens, gt_tokens, smoothfn):
    refs = [gt_tokens]
    hyp  = pred_tokens
    
    # Strictly isolated n-gram precisions to match Lameesa et al. methodology
    b1 = sentence_bleu(refs, hyp, weights=(1.0, 0.0, 0.0, 0.0), smoothing_function=smoothfn)
    b2 = sentence_bleu(refs, hyp, weights=(0.0, 1.0, 0.0, 0.0), smoothing_function=smoothfn)
    b3 = sentence_bleu(refs, hyp, weights=(0.0, 0.0, 1.0, 0.0), smoothing_function=smoothfn)
    b4 = sentence_bleu(refs, hyp, weights=(0.0, 0.0, 0.0, 1.0), smoothing_function=smoothfn)
    
    return b1, b2, b3, b4

def token_f1(pred, gt):
    pred_toks = normalise(pred).split()
    gt_toks   = normalise(gt).split()
    common    = set(pred_toks) & set(gt_toks)
    if not common:
        return 0.0
    p = len(common) / len(pred_toks)
    r = len(common) / len(gt_toks)
    return 2 * p * r / (p + r)

print("Helpers: OK")

# ── Cell 5: Load processor + base model + adapter ──────────────────────────
print("Loading processor …")
processor = AutoProcessor.from_pretrained(MODEL_ID)

# FORCE LEFT PADDING FOR GENERATION
processor.tokenizer.padding_side = "left" 

if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
    print("  Set pad_token = eos_token and padding_side = left")

processor.tokenizer.chat_template = LLAVA_CHAT_TEMPLATE

print("Loading base model (4-bit) …")
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)
base = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb,
    device_map={"": 0},
    low_cpu_mem_usage=True,
    torch_dtype=torch.float16,
)

print("Loading LoRA adapter …")
ft_model = PeftModel.from_pretrained(base, adapter_path)
ft_model.eval()
for param in ft_model.parameters():
    param.requires_grad = False
print("✓ Model ready\n")


# ── Cell 6: Evaluation loop with Unified Semantic Match ────────────────────────
import re
import numpy as np
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# 1. Centralized Independent Evaluator (Run on CPU to safeguard VRAM)
print("Initializing Centralized Evaluator (Standalone BiomedBERT)...")
# Reverted to standard abstract model to match methodology
BIOMEDBERT_ID = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract"
BIO_TOKENIZER = AutoTokenizer.from_pretrained(BIOMEDBERT_ID)
BIO_EMBEDDER  = AutoModel.from_pretrained(BIOMEDBERT_ID).to("cpu")
BIO_EMBEDDER.eval()
print(f"✓ Loaded {BIOMEDBERT_ID}\n")

def compute_unified_semantic_match(pred_str, gt_str, threshold=0.96):
    """Unified evaluation function to eliminate contrastive anisotropy bias."""
    if not pred_str or not gt_str:
        return 0.0, 0
        
    p_clean = re.sub(r"[^\w\s]", " ", str(pred_str).lower()).strip()
    p_clean = re.sub(r"\s+", " ", p_clean)
    g_clean = re.sub(r"[^\w\s]", " ", str(gt_str).lower()).strip()
    g_clean = re.sub(r"\s+", " ", g_clean)
    
    if p_clean == g_clean:
        return 1.0, 1

    inputs = BIO_TOKENIZER([p_clean, g_clean], padding=True, truncation=True, return_tensors="pt").to("cpu")
    
    with torch.no_grad():
        outputs = BIO_EMBEDDER(**inputs)
        attention_mask = inputs["attention_mask"]
        token_embeddings = outputs[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        sum_embeddings = torch.sum(token_embeddings * input_mask_expanded, 1)
        sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9)
        embeddings = sum_embeddings / sum_mask
        embeddings = F.normalize(embeddings, p=2, dim=-1)
        
    sim = (embeddings[0] @ embeddings[1].T).item()
    return sim, int(sim >= threshold)

def print_calibration_report(wrong_sim_scores, threshold):
    if not wrong_sim_scores:
        print("No wrong-answer similarity scores collected.")
        return
    arr = np.array(wrong_sim_scores)
    print("\n" + "=" * 50)
    print("  SEMANTIC-MATCH THRESHOLD CALIBRATION CHECK")
    print("=" * 50)
    print(f"  Threshold in use : {threshold:.4f}")
    print(f"  Wrong-sim min    : {arr.min():.4f}")
    print(f"  Wrong-sim mean   : {arr.mean():.4f}")
    print(f"  Wrong-sim p90    : {np.percentile(arr, 90):.4f}")
    print(f"  Wrong-sim p95    : {np.percentile(arr, 95):.4f}")
    print(f"  Wrong-sim max    : {arr.max():.4f}")
    n_leaking = int((arr >= threshold).sum())
    print(f"  Wrong answers passing threshold: {n_leaking}/{len(arr)} ({n_leaking/len(arr)*100:.1f}%)")
    if n_leaking / len(arr) > 0.02:
        print("  ⚠ More than 2% of wrong answers are passing the semantic-match "
              "threshold. Consider raising the threshold.")
    print("=" * 50)

# 2. Setup Evaluation Data & Variables
val_df   = pd.read_csv(VAL_TXT, sep="|", names=["image_id", "question", "answer"], header=None)
smoothie = SmoothingFunction().method1
results          = []
total            = 0
exact_sum        = 0
semantic_sum     = 0
bleu1_sum = bleu2_sum = bleu3_sum = bleu4_sum = 0.0
token_f1_sum     = 0.0
wrong_sim_scores = []

SEMANTIC_THRESHOLD = 0.96 

print(f"\nEvaluating on {len(val_df)} samples …\n")

# 3. Main Generation Loop
for _, row in tqdm(val_df.iterrows(), total=len(val_df)):
    img_id     = str(row["image_id"]).strip()
    question   = str(row["question"]).strip()
    raw_answer = str(row["answer"]).strip()
    primary_gt = get_synonyms(raw_answer)[0]

    fname    = img_id if img_id.endswith((".jpg", ".png")) else img_id + ".jpg"
    img_path = os.path.join(VAL_IMGS, fname)
    if not os.path.exists(img_path):
        continue

    image  = Image.open(img_path).convert("RGB")
    
    text = f"USER: <image>\n{SYSTEM_PROMPT}\nQuestion: {question}\nASSISTANT:"
    
    inputs = processor(
        text=[text], images=[image], padding=True, return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        gen = ft_model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            pixel_values=inputs.get("pixel_values", None), 
            max_new_tokens=MAX_NEW_TOKENS,
            use_cache=True,
            pad_token_id=processor.tokenizer.pad_token_id,
            eos_token_id=processor.tokenizer.eos_token_id,
            do_sample=False,
        )
        
        trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, gen)]
        pred = normalise(processor.batch_decode(trimmed, skip_special_tokens=True)[0])

    # Metrics
    em = exact_match_any(pred, raw_answer)
    sim_score, sem_m = compute_unified_semantic_match(pred, primary_gt, threshold=SEMANTIC_THRESHOLD)
    
    if em:
        sem_m = 1
        sim_score = max(sim_score, 1.0)
    else:
        wrong_sim_scores.append(sim_score)

    b1, b2, b3, b4 = compute_bleu_ngrams(pred.split(), primary_gt.split(), smoothie)
    tf1 = token_f1(pred, primary_gt)

    total += 1
    exact_sum    += em
    semantic_sum += sem_m
    bleu1_sum += b1; bleu2_sum += b2; bleu3_sum += b3; bleu4_sum += b4
    token_f1_sum += tf1

    results.append({
        "Image ID": img_id, "Question": question,
        "Ground Truth": raw_answer, "Prediction": pred,
        "Exact Match": em,
        "Semantic Match": sem_m,
        "Sim Score": round(sim_score, 4),
        "BLEU-1": round(b1,4), "BLEU-2": round(b2,4),
        "BLEU-3": round(b3,4), "BLEU-4": round(b4,4),
        "Token F1": round(tf1,4),
    })

# 4. Save and Report
results_df = pd.DataFrame(results)
results_df.to_csv(EVAL_CSV, index=False)

# 5. Final Corpus-Level Calculations
avg_b1 = bleu1_sum / total
avg_b2 = bleu2_sum / total
avg_b3 = bleu3_sum / total
avg_b4 = bleu4_sum / total
cumulative_bleu = (avg_b1 * avg_b2 * avg_b3 * avg_b4) ** 0.25

print(f"\nResults → {EVAL_CSV}")
print("\n" + "=" * 57)
print("   LLaVA-Med v1.5 FINE-TUNED — VQA-MED 2021 RESULTS")
print("=" * 57)
print(f"  Checkpoint        : {adapter_path}")
print(f"  Total             : {total}")
print(f"  Exact accuracy    : {exact_sum    / total * 100:.2f}%")
print(f"  Semantic accuracy : {semantic_sum / total * 100:.2f}%")
print(f"  BLEU-1            : {avg_b1 * 100:.2f}")
print(f"  BLEU-2            : {avg_b2 * 100:.2f}")
print(f"  BLEU-3            : {avg_b3 * 100:.2f}")
print(f"  BLEU-4            : {avg_b4 * 100:.2f}")
print(f"  Cumulative BLEU   : {cumulative_bleu * 100:.2f}")
print(f"  Avg Token F1      : {token_f1_sum / total * 100:.2f}")
print("=" * 57)

# Print the calibration report at the very end
print_calibration_report(wrong_sim_scores, threshold=SEMANTIC_THRESHOLD)

HF login: OK
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 101.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatibl

processor_config.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/505 [00:00<?, ?B/s]

The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

... (18 outputs left out)

  0%|          | 0/500 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

100%|██████████| 500/500 [17:05<00:00,  2.05s/it]


Results → /kaggle/working/llava_med_metrics.csv

   LLaVA-Med v1.5 FINE-TUNED — VQA-MED 2021 RESULTS
  Checkpoint        : /kaggle/input/datasets/julfikarrahamanasif/vhcghvfg/checkpoint_epoch3
  Total             : 500
  Exact accuracy    : 43.20%
  Semantic accuracy : 47.00%
  BLEU-1            : 44.71
  BLEU-2            : 33.40
  BLEU-3            : 18.43
  BLEU-4            : 9.98
  Cumulative BLEU   : 22.89
  Avg Token F1      : 45.14

  SEMANTIC-MATCH THRESHOLD CALIBRATION CHECK
  Threshold in use : 0.9600
  Wrong-sim min    : 0.8288
  Wrong-sim mean   : 0.9178
  Wrong-sim p90    : 0.9518
  Wrong-sim p95    : 0.9626
  Wrong-sim max    : 0.9919
  Wrong answers passing threshold: 19/284 (6.7%)
  ⚠ More than 2% of wrong answers are passing the semantic-match threshold. Consider raising the threshold.


In [2]:
# ── Cell 17: Per-class breakdown ──────────────────────────────────────────
results_df["GT_primary"] = results_df["Ground Truth"].apply(
    lambda x: get_synonyms(x)[0] if get_synonyms(x) else x)

class_stats = (
    results_df
    .groupby("GT_primary")
    .agg(
        count=("Exact Match", "count"),
        exact_acc=("Exact Match", "mean"),
        avg_f1=("Token F1", "mean"),
    )
    .sort_values("count", ascending=False)
)

pd.set_option("display.max_rows", 60)
pd.set_option("display.float_format", "{:.3f}".format)
print("\nPer-class accuracy (sorted by count):")
print(class_stats.to_string())

class_csv = EVAL_CSV.replace(".csv", "_per_class.csv")
class_stats.to_csv(class_csv)
print(f"\nPer-class stats → {class_csv}")


Per-class accuracy (sorted by count):
                                                                                count  exact_acc  avg_f1
GT_primary                                                                                              
acute appendicitis                                                                 10      0.600   0.600
neuroblastoma                                                                       9      0.333   0.333
pulmonary embolism                                                                  8      0.500   0.500
horseshoe kidney                                                                    7      0.714   0.714
renal cell carcinoma                                                                7      0.571   0.667
diverticulitis                                                                      6      0.333   0.333
aortic dissection                                                                   6      0.333   0.333
adenocarcinoma o

In [3]:
# ── Cell 16: Sample predictions ───────────────────────────────────────────
print("\n── 20 sample predictions ──")
sample = results_df.sample(min(20, len(results_df)), random_state=42)
for _, r in sample.iterrows():
    mark = "✓" if r["Exact Match"] else "✗"
    print(f"  {mark}  GT: {str(r['Ground Truth']):<40}  "
          f"Pred: {r['Prediction']:<35}  "
          f"F1={r['Token F1']:.2f}")


── 20 sample predictions ──
  ✗  GT: lymphangitic carcinomatosis               Pred: sarcoidosis                          F1=0.00
  ✗  GT: rheumatoid arthritis                      Pred: spine                                F1=0.00
  ✗  GT: odontoid fracture                         Pred: ankylosing spondylitis               F1=0.00
  ✗  GT: intraosseous lipoma                       Pred: osteomyelitis                        F1=0.00
  ✓  GT: pulmonary embolism                        Pred: pulmonary embolism                   F1=1.00
  ✓  GT: motor vehicle accidents and instrument failure  Pred: motor vehicle accidents and instrument failure  F1=1.00
  ✗  GT: round atelectasis                         Pred: pneumonia                            F1=0.00
  ✓  GT: classic bosniak class iii complex renal cyst  Pred: classic bosniak class iii complex renal cyst  F1=1.00
  ✗  GT: congenital cystic adenomatoid malformation  Pred: achalasia                            F1=0.00
  ✓  GT: granular cel